# Proyecto Sprint 12 — Selección de región para nuevos pozos petrolíferos

La compañía OilyGiant necesita determinar en cuál de tres regiones conviene desarrollar 200 nuevos pozos petrolíferos.

El objetivo del proyecto es entrenar modelos de regresión lineal para estimar el volumen de reservas de petróleo, calcular el beneficio potencial de los pozos seleccionados y analizar el riesgo mediante bootstrapping.

Se considerarán las siguientes condiciones de negocio:

- Se estudiarán 500 pozos por región.
- Se seleccionarán los 200 pozos con las mayores reservas predichas.
- El presupuesto para desarrollar los 200 pozos es de 100 millones de dólares.
- Cada unidad de producto representa mil barriles.
- Cada unidad de producto genera 4,500 dólares de ingresos.
- Solo se considerarán regiones con un riesgo de pérdidas inferior al 2.5%.
- Entre las regiones aceptables se elegirá la que tenga el mayor beneficio promedio.

## 1. Preparación y exploración de los datos

Primero se importan las librerías necesarias para cargar y analizar los datos, dividir las muestras, entrenar modelos de regresión lineal y calcular las métricas de error.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

### 1.1 Carga de los datos

Se cargan los datos geológicos correspondientes a las tres regiones disponibles.

In [2]:
geo_0 = pd.read_csv('geo_data_0.csv')
geo_1 = pd.read_csv('geo_data_1.csv')
geo_2 = pd.read_csv('geo_data_2.csv')

### 1.2 Vista inicial de los datos

Se muestran las primeras filas de cada conjunto de datos para conocer su estructura y comprobar las columnas disponibles.

In [3]:
print('REGIÓN 0')
display(geo_0.head())

print('REGIÓN 1')
display(geo_1.head())

print('REGIÓN 2')
display(geo_2.head())

REGIÓN 0


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


REGIÓN 1


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


REGIÓN 2


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


### 1.3 Tipos de datos y valores ausentes

Se revisa el número de observaciones, los tipos de datos y la presencia de valores ausentes en cada región.

In [4]:
print('REGIÓN 0')
geo_0.info()

print('\nREGIÓN 1')
geo_1.info()

print('\nREGIÓN 2')
geo_2.info()

REGIÓN 0
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB

REGIÓN 1
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB

REGIÓN 2
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtyp

### 1.4 Revisión de duplicados

Se comprueba si existen filas completamente duplicadas en alguno de los conjuntos de datos.

In [5]:
print('Duplicados Región 0:', geo_0.duplicated().sum())
print('Duplicados Región 1:', geo_1.duplicated().sum())
print('Duplicados Región 2:', geo_2.duplicated().sum())

Duplicados Región 0: 0
Duplicados Región 1: 0
Duplicados Región 2: 0


También se comprueba si existen identificadores de pozos repetidos. Aunque dos filas no sean idénticas, un mismo identificador podría aparecer más de una vez.

In [6]:
print('IDs duplicados Región 0:', geo_0['id'].duplicated().sum())
print('IDs duplicados Región 1:', geo_1['id'].duplicated().sum())
print('IDs duplicados Región 2:', geo_2['id'].duplicated().sum())

IDs duplicados Región 0: 10
IDs duplicados Región 1: 4
IDs duplicados Región 2: 4


### 1.5 Estadísticas descriptivas

Se revisan las estadísticas básicas de las variables numéricas de cada región, prestando especial atención a `product`, que representa el volumen de reservas del pozo en miles de barriles.

In [7]:
display(geo_0.describe())
display(geo_1.describe())
display(geo_2.describe())

,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.500419,0.250143,2.502647,92.500000
std,0.871832,0.504433,3.248248,44.288691
min,-1.408605,-0.848218,-12.088328,0.000000
25%,-0.072580,-0.200881,0.287748,56.497507
50%,0.502360,0.250252,2.515969,91.849972
75%,1.073581,0.700646,4.715088,128.564089
max,2.362331,1.343769,16.003790,185.364347


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,1.141296,-4.796579,2.494541,68.825000
std,8.965932,5.119872,1.703572,45.944423
min,-31.609576,-26.358598,-0.018144,0.000000
25%,-6.298551,-8.267985,1.000021,26.953261
50%,1.153055,-4.813172,2.011479,57.085625
75%,8.621015,-1.332816,3.999904,107.813044
max,29.421755,18.734063,5.019721,137.945408


,f0,f1,f2,product
count,100000.000000,100000.000000,100000.000000,100000.000000
mean,0.002023,-0.002081,2.495128,95.000000
std,1.732045,1.730417,3.473445,44.749921
min,-8.760004,-7.084020,-11.970335,0.000000
25%,-1.162288,-1.174820,0.130359,59.450441
50%,0.009424,-0.009482,2.484236,94.925613
75%,1.158535,1.163678,4.858794,130.595027
max,7.238262,7.844801,16.739402,190.029838


### Conclusiones de la exploración inicial

Los tres conjuntos de datos contienen 100,000 observaciones y cinco columnas: `id`, `f0`, `f1`, `f2` y `product`.

No se encontraron valores ausentes en ninguna de las tres regiones y los tipos de datos son adecuados para el análisis: `id` es una variable de texto y las demás columnas son numéricas.

Tampoco se encontraron filas completamente duplicadas. Sin embargo, existen algunos identificadores `id` repetidos:

- Región 0: 10 identificadores duplicados.
- Región 1: 4 identificadores duplicados.
- Región 2: 4 identificadores duplicados.

Estos registros no se eliminarán, ya que las filas completas no están duplicadas y la columna `id` no se utilizará como característica para entrenar el modelo.

En cuanto al volumen medio real de reservas (`product`), se observan diferencias entre las regiones:

- Región 0: aproximadamente 92.5 miles de barriles.
- Región 1: aproximadamente 68.8 miles de barriles.
- Región 2: aproximadamente 95.0 miles de barriles.

Por lo tanto, las regiones 0 y 2 presentan mayores reservas promedio que la región 1. Sin embargo, todavía no es posible determinar cuál es la mejor región para desarrollar los nuevos pozos, ya que también será necesario evaluar la precisión de las predicciones, los beneficios potenciales y el riesgo de pérdidas.

La columna `id` se conservará únicamente como identificador de los pozos, pero no se utilizará como característica para entrenar los modelos, ya que no representa una propiedad predictiva del pozo.

## 2. Entrenamiento y evaluación de los modelos

Para cada región se utilizarán las características `f0`, `f1` y `f2` para predecir la variable objetivo `product`.

La columna `id` no se utilizará como característica porque únicamente identifica cada pozo y no representa una propiedad física del mismo.

Los datos de cada región se dividirán en:

- 75% para entrenamiento.
- 25% para validación.

Se utilizará exclusivamente un modelo de regresión lineal, tal como indican las condiciones del proyecto.

La calidad del modelo se evaluará mediante RMSE.

In [8]:
def train_region(data):
    
    # Separamos características y objetivo
    features = data.drop(['id', 'product'], axis=1)
    target = data['product']
    
    # Dividimos los datos:
    # 75% entrenamiento
    # 25% validación
    features_train, features_valid, target_train, target_valid = train_test_split(
        features,
        target,
        test_size=0.25,
        random_state=12345
    )
    
    # Creamos el modelo
    model = LinearRegression()
    
    # Entrenamos el modelo
    model.fit(features_train, target_train)
    
    # Realizamos predicciones
    predictions = model.predict(features_valid)
    
    # Convertimos las predicciones en Series
    predictions = pd.Series(
        predictions,
        index=target_valid.index
    )
    
    # Calculamos la media de las reservas predichas
    predicted_mean = predictions.mean()
    
    # Calculamos RMSE
    rmse = mean_squared_error(
        target_valid,
        predictions
    ) ** 0.5
    
    return predictions, target_valid, predicted_mean, rmse

### 2.1 Entrenamiento y evaluación del modelo de la Región 0

Se aplica la función definida anteriormente a los datos de la Región 0.

El modelo se entrena utilizando el 75% de los datos y se evalúa sobre el 25% restante.

Se analizarán dos resultados principales:

- El volumen promedio de reservas predicho.
- El RMSE, que permite medir qué tan alejadas están las predicciones del modelo respecto a los valores reales.

In [9]:
pred_0, target_0, mean_0, rmse_0 = train_region(geo_0)

print('REGIÓN 0')
print('Reserva promedio predicha:', mean_0)
print('RMSE:', rmse_0)

REGIÓN 0
Reserva promedio predicha: 92.59256778438038
RMSE: 37.5794217150813


### Análisis de la Región 0

El modelo de regresión lineal para la Región 0 predice un volumen promedio de reservas de aproximadamente **92.59 miles de barriles por pozo**.

El RMSE es de aproximadamente **37.58 miles de barriles**. Esto indica que existe una diferencia considerable entre muchas de las predicciones del modelo y los valores reales, por lo que la precisión de las estimaciones individuales es limitada.

Aunque el volumen promedio predicho es relativamente alto, todavía no es posible concluir que esta sea la mejor región. Será necesario comparar sus resultados con los de las otras regiones y, posteriormente, analizar el beneficio potencial y el riesgo de pérdidas.

### 2.2 Entrenamiento y evaluación del modelo de la Región 1

Se aplica el mismo procedimiento a los datos de la Región 1, utilizando el 75% de los datos para entrenamiento y el 25% para validación.

Se calculará nuevamente el volumen promedio de reservas predicho y el RMSE para comparar el rendimiento del modelo con el obtenido en la Región 0.

In [10]:
pred_1, target_1, mean_1, rmse_1 = train_region(geo_1)

print('REGIÓN 1')
print('Reserva promedio predicha:', mean_1)
print('RMSE:', rmse_1)

REGIÓN 1
Reserva promedio predicha: 68.72854689544603
RMSE: 0.8930992867756166


### Análisis de la Región 1

El modelo de regresión lineal para la Región 1 predice un volumen promedio de reservas de aproximadamente **68.73 miles de barriles por pozo**.

El RMSE es de aproximadamente **0.89 miles de barriles**, un valor mucho menor que el obtenido para la Región 0. Esto indica que el modelo realiza predicciones considerablemente más precisas para esta región.

Aunque la Región 1 presenta un volumen promedio de reservas menor que la Región 0, la incertidumbre asociada a las predicciones del modelo es mucho menor. Esta mayor precisión podría ser relevante al evaluar posteriormente el riesgo económico.

Todavía no es posible decidir cuál región es la mejor, ya que falta analizar la Región 2 y posteriormente comparar el beneficio potencial y el riesgo de pérdidas.

### 2.3 Entrenamiento y evaluación del modelo de la Región 2

Se aplica el mismo procedimiento a los datos de la Región 2.

Se calcularán el volumen promedio de reservas predicho y el RMSE para completar la comparación de los modelos de las tres regiones.

In [11]:
pred_2, target_2, mean_2, rmse_2 = train_region(geo_2)

print('REGIÓN 2')
print('Reserva promedio predicha:', mean_2)
print('RMSE:', rmse_2)

REGIÓN 2
Reserva promedio predicha: 94.96504596800492
RMSE: 40.02970873393434


### Análisis de la Región 2

El modelo de regresión lineal para la Región 2 predice un volumen promedio de reservas de aproximadamente **94.97 miles de barriles** por pozo.

El RMSE es de aproximadamente **40.03 miles de barriles**, el valor más alto de las tres regiones. Esto indica que, aunque la Región 2 presenta el mayor volumen promedio de reservas predicho, las predicciones individuales tienen una incertidumbre considerable.

Por lo tanto, un promedio alto de reservas no garantiza por sí solo que esta sea la mejor región para invertir. Será necesario considerar también la rentabilidad esperada y el riesgo de pérdidas.

### 2.4 Comparación de los modelos de las tres regiones

Los resultados obtenidos son los siguientes:

| Región | Reserva promedio predicha | RMSE |
|---|---:|---:|
| Región 0 | 92.59 | 37.58 |
| Región 1 | 68.73 | 0.89 |
| Región 2 | 94.97 | 40.03 |

La Región 2 presenta el mayor volumen promedio predicho, seguida de la Región 0. Sin embargo, ambas regiones tienen valores de RMSE elevados, lo que indica una mayor incertidumbre en sus predicciones.

La Región 1 presenta un volumen promedio predicho menor, pero su RMSE es mucho más bajo. Esto significa que el modelo predice los valores reales de esta región con mucha mayor precisión.

En esta etapa todavía no se debe seleccionar una región únicamente con base en el promedio de reservas o el RMSE. La decisión final dependerá del beneficio potencial y del riesgo de pérdidas bajo las condiciones económicas del proyecto.

## 3. Preparación para el cálculo de ganancias

A continuación se definen las condiciones económicas del proyecto.

La empresa dispone de un presupuesto total de **100 millones de dólares** para desarrollar **200 pozos**.

Cada unidad de `product` representa mil barriles de petróleo y genera **4,500 dólares de ingresos**.

También se calculará el volumen mínimo promedio que debe producir cada pozo para que la inversión no genere pérdidas.

In [12]:
BUDGET = 100_000_000
WELLS_COUNT = 200
REVENUE_PER_UNIT = 4500

budget_per_well = BUDGET / WELLS_COUNT
break_even_units = budget_per_well / REVENUE_PER_UNIT

print('Presupuesto por pozo:', budget_per_well)
print('Reservas mínimas por pozo para no tener pérdidas:', break_even_units)

Presupuesto por pozo: 500000.0
Reservas mínimas por pozo para no tener pérdidas: 111.11111111111111


### 3.1 Punto de equilibrio y comparación con las reservas promedio

El presupuesto total es de **100 millones de dólares** para desarrollar **200 pozos**, por lo que la inversión promedio disponible por pozo es de:

**500,000 dólares por pozo**

Como cada unidad de `product` genera **4,500 dólares**, el volumen promedio mínimo necesario para recuperar la inversión es de aproximadamente:

**111.11 unidades de producto por pozo**

Como cada unidad representa mil barriles, esto equivale aproximadamente a **111.11 mil barriles por pozo**.

Comparando este valor con las reservas promedio predichas:

| Región | Reserva promedio predicha | Punto de equilibrio |
|---|---:|---:|
| Región 0 | 92.59 | 111.11 |
| Región 1 | 68.73 | 111.11 |
| Región 2 | 94.97 | 111.11 |

Las reservas promedio predichas de las tres regiones se encuentran por debajo del punto de equilibrio de 111.11 unidades por pozo.

Sin embargo, esto no significa que el proyecto sea necesariamente inviable, ya que la empresa no desarrollará pozos elegidos al azar. De los puntos estudiados se seleccionarán únicamente los **200 pozos con las mayores reservas predichas**.

Por lo tanto, el siguiente paso será seleccionar los pozos más prometedores y calcular el beneficio utilizando las reservas reales de esos mismos pozos.

## 4. Cálculo del beneficio de los pozos seleccionados

Se creará una función que:

1. Ordene los pozos según las reservas predichas por el modelo.
2. Seleccione los 200 pozos con las predicciones más altas.
3. Consulte las reservas reales de esos mismos pozos.
4. Calcule los ingresos generados.
5. Reste el presupuesto total de desarrollo para obtener el beneficio.

In [13]:
def calculate_profit(target, predictions, count):
    
    predictions_sorted = predictions.sort_values(ascending=False)
    
    selected_indexes = predictions_sorted.head(count).index
    
    selected_target = target.loc[selected_indexes]
    
    revenue = selected_target.sum() * REVENUE_PER_UNIT
    
    profit = revenue - BUDGET
    
    return profit

### 4.1 Selección de los 200 mejores pozos de cada región

A partir de las predicciones realizadas sobre los conjuntos de validación, se seleccionan los **200 pozos con las mayores reservas predichas** para cada región.

Además de guardar las predicciones de estos pozos, se obtienen las reservas reales correspondientes. Esto permite comparar la estimación del modelo con el volumen real de reservas utilizado posteriormente para calcular el beneficio.

In [14]:
top_200_pred_0 = pred_0.sort_values(ascending=False).head(WELLS_COUNT)
top_200_pred_1 = pred_1.sort_values(ascending=False).head(WELLS_COUNT)
top_200_pred_2 = pred_2.sort_values(ascending=False).head(WELLS_COUNT)

top_200_target_0 = target_0.loc[top_200_pred_0.index]
top_200_target_1 = target_1.loc[top_200_pred_1.index]
top_200_target_2 = target_2.loc[top_200_pred_2.index]

print('Reservas predichas totales Región 0:', top_200_pred_0.sum())
print('Reservas reales totales Región 0:', top_200_target_0.sum())

print('\nReservas predichas totales Región 1:', top_200_pred_1.sum())
print('Reservas reales totales Región 1:', top_200_target_1.sum())

print('\nReservas predichas totales Región 2:', top_200_pred_2.sum())
print('Reservas reales totales Región 2:', top_200_target_2.sum())

Reservas predichas totales Región 0: 31102.330838811387
Reservas reales totales Región 0: 29601.83565142189

Reservas predichas totales Región 1: 27746.026782163477
Reservas reales totales Región 1: 27589.081548181137

Reservas predichas totales Región 2: 29603.898658318365
Reservas reales totales Región 2: 28245.22214133296


### Análisis de los 200 pozos seleccionados

Los 200 pozos seleccionados en cada región corresponden a aquellos con las mayores reservas predichas por el modelo.

La Región 0 presenta el mayor volumen total predicho y también el mayor volumen real entre los 200 pozos seleccionados.

La Región 1 muestra una diferencia pequeña entre las reservas predichas y las reales, lo que es coherente con el bajo RMSE observado anteriormente.

La Región 2 también presenta un volumen considerable de reservas, aunque sus predicciones muestran una mayor diferencia respecto a los valores reales.

A continuación se utilizarán las reservas reales de estos pozos seleccionados para calcular el beneficio potencial de cada región.

In [15]:
profit_0 = calculate_profit( target_0, pred_0, WELLS_COUNT)

print('Beneficio potencial de la Región 0:', profit_0)

Beneficio potencial de la Región 0: 33208260.43139851


### 4.2 Beneficio potencial de la Región 0

Al seleccionar los 200 pozos con las mayores reservas predichas en la Región 0, se obtiene un beneficio potencial de aproximadamente **33.21 millones de dólares**.

Este resultado indica que, para esta selección concreta de pozos, los ingresos generados superarían el presupuesto inicial de 100 millones de dólares.

Sin embargo, este cálculo utiliza los mejores pozos de todo el conjunto de validación y todavía no incorpora la restricción real de estudiar únicamente 500 puntos ni el riesgo asociado a esa selección.

In [16]:
profit_1 = calculate_profit( target_1, pred_1, WELLS_COUNT)

print('Beneficio potencial de la Región 1:', profit_1)

Beneficio potencial de la Región 1: 24150866.966815114


### 4.3 Beneficio potencial de la Región 1

Al seleccionar los 200 pozos con las mayores reservas predichas en la Región 1, se obtiene un beneficio potencial de aproximadamente **24.15 millones de dólares**.

El beneficio es positivo, por lo que los ingresos generados por estos pozos superarían el presupuesto inicial de 100 millones de dólares.

Este beneficio es menor que el obtenido para la Región 0. No obstante, todavía falta considerar la variabilidad y el riesgo asociados al proceso real de exploración.

In [17]:
profit_2 = calculate_profit( target_2, pred_2, WELLS_COUNT)

print('Beneficio potencial de la Región 2:', profit_2)

Beneficio potencial de la Región 2: 27103499.635998324


### 4.4 Beneficio potencial de la Región 2

Al seleccionar los 200 pozos con las mayores reservas predichas en la Región 2, se obtiene un beneficio potencial de aproximadamente **27.10 millones de dólares**.

Este valor es superior al de la Región 1, pero inferior al de la Región 0.

Al igual que en las regiones anteriores, este resultado representa una estimación puntual y todavía no considera el riesgo asociado a seleccionar únicamente 500 pozos durante la exploración.

### 4.5 Comparación preliminar del beneficio potencial

Los beneficios potenciales obtenidos al seleccionar los 200 pozos con mayores reservas predichas son:

| Región | Beneficio potencial aproximado |
|---|---:|
| Región 0 | 33.21 millones de dólares |
| Región 1 | 24.15 millones de dólares |
| Región 2 | 27.10 millones de dólares |

Si la decisión se tomara únicamente con base en este cálculo puntual, la **Región 0** sería la opción preliminar, ya que presenta el mayor beneficio potencial.

Sin embargo, esta todavía no es la decisión definitiva. En la situación real del negocio solo se estudiarán 500 pozos y, de esos, se seleccionarán los 200 mejores.

Por esta razón, el siguiente paso será utilizar bootstrapping para simular repetidamente este proceso y evaluar no solo el beneficio promedio, sino también la incertidumbre y el riesgo de pérdidas.

## 5. Análisis de beneficios y riesgos mediante bootstrapping

Para simular el proceso real de exploración, en cada iteración se seleccionarán aleatoriamente **500 pozos** del conjunto de validación.

De esos 500 pozos se elegirán los **200 con las mayores reservas predichas** y se calculará el beneficio utilizando las reservas reales.

Este proceso se repetirá **1000 veces** para construir una distribución de posibles beneficios.

A partir de esta distribución se calcularán:

- El beneficio promedio.
- El intervalo de confianza del 95%.
- El riesgo de pérdidas.

In [18]:
SAMPLE_SIZE = 500
BOOTSTRAP_SAMPLES = 1000

### 5.1 Procedimiento de bootstrapping

Debido a que el muestreo bootstrap se realiza con reemplazo, una misma observación puede aparecer varias veces dentro de una submuestra.

Para mantener correctamente alineados los valores reales y las predicciones, ambos se combinarán en una sola tabla antes de realizar el remuestreo. Después de seleccionar los 500 pozos de cada simulación, se reiniciarán los índices de la submuestra.

El procedimiento se implementará en una función para aplicar exactamente el mismo proceso a las tres regiones y evitar duplicación de código.

In [19]:
def bootstrap_profit(target, predictions):
    
    state = np.random.RandomState(12345)
    values = []
    
    # Juntamos valor real y predicción
    data_bootstrap = pd.DataFrame({
        'target': target,
        'prediction': predictions
    })
    
    for i in range(BOOTSTRAP_SAMPLES):
        
        # Seleccionamos 500 pozos con reemplazo
        subsample = data_bootstrap.sample(
            n=SAMPLE_SIZE,
            replace=True,
            random_state=state
        ).reset_index(drop=True)
        
        # Calculamos el beneficio de los mejores 200
        profit = calculate_profit(
            subsample['target'],
            subsample['prediction'],
            WELLS_COUNT
        )
        
        values.append(profit)
    
    values = pd.Series(values)
    
    mean_profit = values.mean()
    lower = values.quantile(0.025)
    upper = values.quantile(0.975)
    risk = (values < 0).mean() * 100
    
    return values, mean_profit, lower, upper, risk

In [20]:
values_0, mean_profit_0, lower_0, upper_0, risk_0 = bootstrap_profit(target_0,pred_0)

print('Beneficio promedio Región 0:', mean_profit_0)
print('Intervalo de confianza del 95%:', lower_0, upper_0)
print('Riesgo de pérdidas Región 0:', risk_0, '%')

Beneficio promedio Región 0: 3961649.8480237117
Intervalo de confianza del 95%: -1112155.4589049604 9097669.41553423
Riesgo de pérdidas Región 0: 6.9 %


### 5.2 Análisis del bootstrapping de la Región 0

La Región 0 presenta un beneficio promedio estimado de aproximadamente **3.96 millones de dólares**.

El intervalo de confianza del 95% para el beneficio se encuentra aproximadamente entre **-1.12 millones y 9.10 millones de dólares**.

El límite inferior del intervalo es negativo, lo que indica que existen escenarios plausibles en los que el desarrollo de los pozos podría generar pérdidas.

El riesgo estimado de pérdidas es de **6.9%**, valor superior al límite máximo permitido del **2.5%** establecido por la empresa.

Por lo tanto, la Región 0 no cumple con el criterio de riesgo y no puede considerarse una candidata válida para la selección final.

In [21]:
values_1, mean_profit_1, lower_1, upper_1, risk_1 = bootstrap_profit(target_1,pred_1)

print('Beneficio promedio Región 1:', mean_profit_1)
print('Intervalo de confianza del 95%:', lower_1, upper_1)
print('Riesgo de pérdidas Región 1:', risk_1, '%')

Beneficio promedio Región 1: 4560451.057866608
Intervalo de confianza del 95%: 338205.0939898458 8522894.538660347
Riesgo de pérdidas Región 1: 1.5 %


### 5.3 Análisis del bootstrapping de la Región 1

La Región 1 presenta un beneficio promedio estimado de aproximadamente **4.56 millones de dólares**.

El intervalo de confianza del 95% para el beneficio se encuentra aproximadamente entre **0.34 millones y 8.52 millones de dólares**.

El límite inferior del intervalo es positivo. Además, únicamente el **1.5%** de las simulaciones generaron pérdidas, lo que indica una baja probabilidad estimada de obtener un beneficio negativo bajo las condiciones simuladas.

El riesgo estimado de pérdidas es de **1.5%**, valor inferior al límite máximo permitido del **2.5%**.

Por lo tanto, la Región 1 cumple con el criterio de riesgo establecido por la empresa y continúa siendo una candidata válida para la selección final.

In [22]:
values_2, mean_profit_2, lower_2, upper_2, risk_2 = bootstrap_profit(target_2,pred_2)

print('Beneficio promedio Región 2:', mean_profit_2)
print('Intervalo de confianza del 95%:', lower_2, upper_2)
print('Riesgo de pérdidas Región 2:', risk_2, '%')

Beneficio promedio Región 2: 4044038.665683568
Intervalo de confianza del 95%: -1633504.1339559986 9503595.749237997
Riesgo de pérdidas Región 2: 7.6 %


### 5.4 Análisis del bootstrapping de la Región 2

La Región 2 presenta un beneficio promedio estimado de aproximadamente **4.04 millones de dólares**.

El intervalo de confianza del 95% para el beneficio se encuentra aproximadamente entre **-1.63 millones y 9.50 millones de dólares**.

El límite inferior del intervalo es negativo, lo que indica que existen escenarios plausibles en los que la inversión podría producir pérdidas.

El riesgo estimado de pérdidas es de **7.6%**, valor superior al límite máximo permitido del **2.5%**.

Por lo tanto, la Región 2 no cumple con el criterio de riesgo establecido por la empresa y queda descartada para la selección final.

## 6. Comparación final y selección de la región

Los resultados del análisis mediante bootstrapping son:

| Región | Beneficio promedio | Intervalo de confianza del 95% | Riesgo de pérdidas | Cumple riesgo < 2.5% |
|---|---:|---:|---:|---|
| Región 0 | 3.96 millones USD | -1.12 a 9.10 millones USD | 6.9% | No |
| Región 1 | 4.56 millones USD | 0.34 a 8.52 millones USD | 1.5% | Sí |
| Región 2 | 4.04 millones USD | -1.63 a 9.50 millones USD | 7.6% | No |

Según las condiciones del proyecto, primero deben descartarse todas las regiones cuyo riesgo de pérdidas sea igual o superior al 2.5%.

Las Regiones 0 y 2 no cumplen este requisito, con riesgos de pérdidas de 6.9% y 7.6%, respectivamente.

La Región 1 presenta un riesgo de pérdidas de **1.5%**, por debajo del límite establecido, por lo que es la única región que supera el filtro de riesgo.

Entre las regiones que cumplen el criterio de riesgo, la Región 1 presenta un beneficio promedio estimado de aproximadamente **4.56 millones de dólares**.

Por lo tanto, la **Región 1** es la región recomendada para el desarrollo de los 200 nuevos pozos petrolíferos.

### 6.1 Comparación con la selección preliminar

La selección final no coincide con la elección preliminar basada únicamente en el beneficio potencial de los 200 mejores pozos de todo el conjunto de validación.

En la comparación preliminar, la Región 0 presentaba el mayor beneficio potencial, con aproximadamente **33.21 millones de dólares**. Sin embargo, el análisis mediante bootstrapping mostró que su riesgo de pérdidas es de **6.9%**, superior al máximo permitido del **2.5%**.

La Región 1, aunque presentaba un beneficio potencial menor en la comparación inicial, ofrece una combinación más favorable de rentabilidad y riesgo. Su riesgo de pérdidas es de solo **1.5%**, por lo que es la única región que cumple con las condiciones del negocio.

Esto demuestra por qué no es suficiente seleccionar una región únicamente por su beneficio potencial máximo; también es necesario considerar la incertidumbre y el riesgo asociados al proceso de selección de los pozos.

## Conclusión general

Se analizaron los datos geológicos de tres regiones y se entrenó un modelo de regresión lineal para cada una con el objetivo de predecir el volumen de reservas de petróleo.

Los modelos mostraron comportamientos diferentes. La Región 1 presentó el menor volumen promedio predicho, pero también un RMSE considerablemente menor, lo que indica predicciones mucho más precisas que en las Regiones 0 y 2.

El punto de equilibrio del proyecto es de aproximadamente **111.11 unidades de producto por pozo**, valor superior al promedio general predicho de las tres regiones. Sin embargo, el proyecto selecciona únicamente los pozos con mayores reservas predichas, por lo que se evaluaron los 200 mejores pozos de cada región.

En una comparación preliminar, la Región 0 presentó el mayor beneficio potencial. No obstante, al simular el proceso real de exploración mediante bootstrapping con 1000 muestras de 500 pozos, los resultados mostraron diferencias importantes en el riesgo.

Los resultados finales fueron:

- Región 0: beneficio promedio de aproximadamente **3.96 millones de dólares** y riesgo de pérdidas de **6.9%**.
- Región 1: beneficio promedio de aproximadamente **4.56 millones de dólares** y riesgo de pérdidas de **1.5%**.
- Región 2: beneficio promedio de aproximadamente **4.04 millones de dólares** y riesgo de pérdidas de **7.6%**.

Las Regiones 0 y 2 superan el límite máximo de riesgo permitido del 2.5%, por lo que deben descartarse.

Por lo tanto, la **Región 1** es la opción recomendada para el desarrollo de los 200 nuevos pozos petrolíferos, ya que es la única que cumple con el criterio de riesgo y además presenta el mayor beneficio promedio entre las regiones aceptables.